# Module 7 — Exercise Solutions
**Nutanix AI/ML Intermediate Workshop**

Complete solutions for all 10 exercises in `Lab_7_Exercises.ipynb`.
Each section explains the approach, then provides a fully working solution cell.

> **Context:** Nutanix CVM telemetry • IsolationForest anomaly detection • Gemini API remediation • FastAPI service • JSONL audit log

## Shared Imports and Setup
Run this cell first — every exercise solution depends on these imports and constants.

In [27]:
import numpy as np
import pandas as pd
import json
import pathlib
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Optional

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from pydantic import BaseModel

# Feature list used across exercises 3, 4, 7, 8, 10
FEATURES = [
    'cpu_pct', 'mem_pct', 'iops', 'latency_ms',
    'disk_read_mbps', 'disk_write_mbps',
    'network_mbps', 'stargate_ops_per_sec',
    'disk_pressure', 'network_cpu_ratio',
]

print('Shared imports loaded.')

Shared imports loaded.


---
## Exercise 1 — Telemetry Generation
**Difficulty:** ⭐ Beginner

### Approach
`generate_scenario` uses a lookup table keyed by scenario name. Each entry holds
the `(low, high)` range for every metric that varies by scenario. The three
additional disk and network columns use fixed realistic ranges that are the same
across all scenarios — only CPU, memory, IOPS, latency, and network throughput
differ meaningfully between `normal`, `cpu_spike`, and `network_storm`.

`np.random.uniform(lo, hi, n)` is used for every column so that each row is
independently sampled, giving a realistic spread within each scenario's bounds.
A fixed `np.random.seed(42)` is set at the start so results are reproducible.

In [28]:
def generate_scenario(scenario: str, n: int) -> pd.DataFrame:
    """
    Generate n rows of CVM telemetry for the given scenario.

    Scenarios
    ---------
    normal        : baseline healthy CVM operation
    cpu_spike     : CPU overcommit / VM migration storm
    network_storm : broadcast / LACP / OVS storm
    """
    np.random.seed(42)
    nodes = [f'node-{i}' for i in range(1, 5)]

    # (low, high) per scenario for variable columns
    ranges = {
        'normal': dict(
            cpu=(20, 50),    mem=(40, 65),   iops=(5_000, 12_000),
            lat=(1, 5),      net=(200, 800), sops=(1_500, 2_500),
        ),
        'cpu_spike': dict(
            cpu=(85, 99),    mem=(75, 95),   iops=(15_000, 25_000),
            lat=(30, 80),    net=(400, 900), sops=(200, 600),
        ),
        'network_storm': dict(
            cpu=(50, 70),    mem=(50, 70),   iops=(6_000, 10_000),
            lat=(20, 50),    net=(1_500, 2_500), sops=(1_200, 2_000),
        ),
    }

    if scenario not in ranges:
        raise ValueError(f"Unknown scenario '{scenario}'. Choose from: {list(ranges)}")

    r = ranges[scenario]

    def rnd(lo, hi):
        return np.random.uniform(lo, hi, n)

    return pd.DataFrame({
        'node':                 np.random.choice(nodes, n),
        'cpu_pct':              rnd(*r['cpu']),
        'mem_pct':              rnd(*r['mem']),
        'iops':                 rnd(*r['iops']),
        'latency_ms':           rnd(*r['lat']),
        'disk_read_mbps':       rnd(80, 200),
        'disk_write_mbps':      rnd(50, 150),
        'network_mbps':         rnd(*r['net']),
        'stargate_ops_per_sec': rnd(*r['sops']),
    })


# ── Acceptance test ───────────────────────────────────────────────────────
df_spike = generate_scenario('cpu_spike', 50)
assert df_spike.shape[0] == 50, "Must return 50 rows"
assert df_spike['cpu_pct'].mean() > 85, (
    f"cpu_pct mean should be >85, got {df_spike['cpu_pct'].mean():.1f}"
)
print('Exercise 1 passed ✅')
print(df_spike[['node', 'cpu_pct', 'mem_pct', 'iops', 'latency_ms']].describe().round(1))

Exercise 1 passed ✅
       cpu_pct  mem_pct     iops  latency_ms
count     50.0     50.0     50.0        50.0
mean      91.8     84.8  19608.3        55.1
std        4.4      5.8   2833.3        14.7
min       85.1     75.5  15069.5        30.3
25%       87.8     79.7  17401.9        41.9
50%       92.1     85.0  19582.8        58.1
75%       96.0     90.2  21653.9        66.1
max       98.8     93.6  24856.5        76.8


---
## Exercise 2 — Feature Engineering
**Difficulty:** ⭐⭐ Intermediate

### Approach
Both derived features are simple ratio transforms that compress multiple raw
columns into single diagnostic signals:

- **`disk_pressure`** = `(disk_read_mbps + disk_write_mbps) / stargate_ops_per_sec`  
  A high value means each Stargate operation is doing disproportionate disk I/O,
  indicating storage path stress or WAL replay.
  A small epsilon (`+ 0.001`) prevents division-by-zero when Stargate is down.

- **`network_cpu_ratio`** = `network_mbps / (cpu_pct + 1)`  
  Elevated values flag network activity that is not proportional to CPU work —
  the classic signature of a network storm or broadcast amplification.
  `+ 1` avoids division-by-zero when cpu_pct is 0.

The function works on a copy of the input so the caller's DataFrame is not mutated.

In [29]:
def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add two engineered diagnostic features to a CVM telemetry DataFrame.

    disk_pressure      : disk throughput per Stargate op (higher = more I/O stress)
    network_cpu_ratio  : network load relative to CPU (flags network storms)
    """
    df = df.copy()
    # Epsilon guard prevents division-by-zero when stargate_ops_per_sec is 0
    df['disk_pressure']     = (
        (df['disk_read_mbps'] + df['disk_write_mbps'])
        / (df['stargate_ops_per_sec'] + 0.001)
    )
    # +1 guard prevents division-by-zero when cpu_pct is 0
    df['network_cpu_ratio'] = df['network_mbps'] / (df['cpu_pct'] + 1)
    return df


# ── Acceptance test ───────────────────────────────────────────────────────
df_normal = generate_scenario('normal', 100)
df_feat   = add_custom_features(df_normal)

assert 'disk_pressure'     in df_feat.columns, 'disk_pressure column missing'
assert 'network_cpu_ratio' in df_feat.columns, 'network_cpu_ratio column missing'
assert df_feat['disk_pressure'].isna().sum() == 0,     'disk_pressure has NaN'
assert df_feat['network_cpu_ratio'].isna().sum() == 0, 'network_cpu_ratio has NaN'

# Original DataFrame must be unchanged
assert 'disk_pressure' not in df_normal.columns, 'Original DataFrame was mutated'

print('Exercise 2 passed ✅')
print(df_feat[['disk_pressure', 'network_cpu_ratio']].describe().round(4))

Exercise 2 passed ✅
       disk_pressure  network_cpu_ratio
count       100.0000           100.0000
mean          0.1222            14.8401
std           0.0304             6.8777
min           0.0628             4.9355
25%           0.1000             9.7307
50%           0.1175            13.4352
75%           0.1347            18.1464
max           0.2117            33.4626


---
## Exercise 3 — Anomaly Detection: Contamination Comparison
**Difficulty:** ⭐⭐ Intermediate

### Approach
`IsolationForest`'s `contamination` parameter sets the decision threshold: the
model scores every sample and then marks the bottom `contamination × 100%` as
anomalies. Changing it does **not** retrain the forest — it only shifts the
cut-off score (the `offset_` attribute).

The test set is deliberately imbalanced: 150 normal rows + 50 cpu_spike rows,
giving a ground-truth anomaly rate of 25%. We expect:
- `model_strict` (2%) to flag roughly 4 rows — only the most extreme outliers.
- `model_loose` (10%) to flag roughly 20 rows — catching all of the spike rows
  and some borderline normal ones.

Both models are fit on the same scaled data so the comparison is fair.
The `StandardScaler` is fit once and reused by both models and all later exercises.

In [30]:
# ── Build test dataset ────────────────────────────────────────────────────
df_test = pd.concat([
    generate_scenario('normal',    150),
    generate_scenario('cpu_spike',  50),
], ignore_index=True)

df_test = add_custom_features(df_test)

X        = df_test[FEATURES].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Train both models ─────────────────────────────────────────────────────
model_strict = IsolationForest(contamination=0.02, n_estimators=100, random_state=42)
model_loose  = IsolationForest(contamination=0.10, n_estimators=100, random_state=42)

model_strict.fit(X_scaled)
model_loose.fit(X_scaled)

strict_preds = model_strict.predict(X_scaled)
loose_preds  = model_loose.predict(X_scaled)

strict_anomalies = (strict_preds == -1).sum()
loose_anomalies  = (loose_preds  == -1).sum()

print(f'model_strict (2%)  flagged: {strict_anomalies:3d} anomalies '
      f'({strict_anomalies / len(X) * 100:.1f}%)')
print(f'model_loose  (10%) flagged: {loose_anomalies:3d} anomalies '
      f'({loose_anomalies / len(X) * 100:.1f}%)')
print(f'\nRatio loose/strict: {loose_anomalies / max(strict_anomalies, 1):.1f}x')

# ── Decision boundary comparison ─────────────────────────────────────────
print(f'\nmodel_strict offset_ : {model_strict.offset_:.4f}')
print(f'model_loose  offset_ : {model_loose.offset_:.4f}')
print('(Lower offset = stricter threshold = fewer anomalies flagged)')

assert loose_anomalies > strict_anomalies, 'Loose model should flag more anomalies'
print('\nExercise 3 passed ✅')

model_strict (2%)  flagged:   4 anomalies (2.0%)
model_loose  (10%) flagged:  20 anomalies (10.0%)

Ratio loose/strict: 5.0x

model_strict offset_ : -0.5822
model_loose  offset_ : -0.5569
(Lower offset = stricter threshold = fewer anomalies flagged)

Exercise 3 passed ✅


In [31]:
# ── Retrain on NORMAL-ONLY data for pipeline exercises (7 – 10) ──────────
#
# Cell 8 uses mixed data to demonstrate contamination rate comparison.
# For the actual pipeline (mini_pipeline, parallel batch, RAG, integration
# test) the model must be trained on *only* healthy CVM data so that
# anomalous nodes produce reliably low scores.
#
df_normal_prod  = add_custom_features(generate_scenario('normal', 200))
X_norm_prod     = df_normal_prod[FEATURES].fillna(0)

scaler          = StandardScaler()
X_norm_scaled   = scaler.fit_transform(X_norm_prod)

model_strict    = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
model_loose     = IsolationForest(contamination=0.15, n_estimators=100, random_state=42)

model_strict.fit(X_norm_scaled)
model_loose.fit(X_norm_scaled)

print('Production models retrained on normal-only data (exercises 7–10):')
print(f'  model_strict  contamination=5%   offset_={model_strict.offset_:.4f}')
print(f'  model_loose   contamination=15%  offset_={model_loose.offset_:.4f}')
print('These override the mixed-data models from Exercise 3.')
print('Normal CVMs will score above offset_; anomalous CVMs will score below.')


Production models retrained on normal-only data (exercises 7–10):
  model_strict  contamination=5%   offset_=-0.5450
  model_loose   contamination=15%  offset_=-0.5256
These override the mixed-data models from Exercise 3.
Normal CVMs will score above offset_; anomalous CVMs will score below.


---
## Exercise 4 — FastAPI Batch Detect Endpoint
**Difficulty:** ⭐⭐ Intermediate

### Approach
The endpoint accepts a single query-string parameter `nodes` — a comma-separated
list of node names. This is a common REST pattern for lightweight batch requests
that avoids the overhead of a POST body while still keeping the URL readable.

The implementation:
1. Splits `nodes` on commas and strips whitespace from each name.
2. For each node, constructs a feature vector using preset "normal" baseline
   values plus the two engineered features. In a production API this would
   instead call a time-series store like Prometheus or Nutanix Prism.
3. Scales the vector with the same `StandardScaler` fitted in Exercise 3.
4. Calls `model_strict.predict` and `model_strict.score_samples` to get both
   the binary label and the raw anomaly score.
5. Derives a human-readable `severity` string and returns a list of dicts.

The Pydantic models below are included to show how this would look wired into
a real FastAPI app with `@app.get('/detect/batch')`.

In [32]:
# ── Pydantic schemas ──────────────────────────────────────────────────────
class DetectionResponse(BaseModel):
    node:          str
    is_anomaly:    bool
    anomaly_score: float
    severity:      str


# Baseline "healthy" feature values used when live metrics are unavailable
NORMAL_METRICS: Dict[str, float] = dict(
    cpu_pct=35.0,         mem_pct=55.0,         iops=8_000.0,
    latency_ms=2.5,       disk_read_mbps=120.0, disk_write_mbps=80.0,
    network_mbps=500.0,   stargate_ops_per_sec=2_000.0,
    disk_pressure=0.10,   network_cpu_ratio=13.9,
)


def detect_batch_logic(nodes_str: str) -> List[Dict]:
    """
    Core logic for GET /detect/batch?nodes=node-1,node-2,...

    Splits the comma-separated node list, runs anomaly detection on preset
    normal baseline metrics for each node, and returns a list of detection
    result dicts that FastAPI would serialize to JSON.
    """
    nodes   = [n.strip() for n in nodes_str.split(',') if n.strip()]
    results = []

    for node in nodes:
        # In production: fetch live metrics from Prism / Prometheus here
        row   = NORMAL_METRICS.copy()
        X_row = np.array([[row[f] for f in FEATURES]])
        X_s   = scaler.transform(X_row)

        score  = float(model_strict.score_samples(X_s)[0])
        is_an  = bool(model_strict.predict(X_s)[0] == -1)
        sev    = 'critical' if is_an and score < model_strict.offset_ * 1.5 else \
                 'warning'  if is_an else 'none'

        results.append(DetectionResponse(
            node          = node,
            is_anomaly    = is_an,
            anomaly_score = round(score, 5),
            severity      = sev,
        ).model_dump())

    return results


# ── Illustrative FastAPI route (not executed here) ────────────────────────
FASTAPI_ROUTE_CODE = '''
from fastapi import FastAPI
app = FastAPI()

@app.get("/detect/batch")
def detect_batch(nodes: str = "node-1,node-2,node-3") -> List[DetectionResponse]:
    return detect_batch_logic(nodes)
'''
print('FastAPI route definition:')
print(FASTAPI_ROUTE_CODE)

# ── Acceptance test ───────────────────────────────────────────────────────
results = detect_batch_logic('node-1,node-2,node-3')
assert len(results) == 3, 'Should return 3 results'
assert all('node' in r and 'is_anomaly' in r for r in results)
assert all(r['severity'] in {'critical', 'warning', 'none'} for r in results)

print('Exercise 4 passed ✅')
for r in results:
    print(f"  {r['node']:8} — anomaly={r['is_anomaly']}  "
          f"score={r['anomaly_score']:.5f}  severity={r['severity']}")

FastAPI route definition:

from fastapi import FastAPI
app = FastAPI()

@app.get("/detect/batch")
def detect_batch(nodes: str = "node-1,node-2,node-3") -> List[DetectionResponse]:
    return detect_batch_logic(nodes)

Exercise 4 passed ✅
  node-1   — anomaly=False  score=-0.43507  severity=none
  node-2   — anomaly=False  score=-0.43507  severity=none
  node-3   — anomaly=False  score=-0.43507  severity=none


/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


---
## Exercise 5 — LLM Remediation Quality Scorer
**Difficulty:** ⭐⭐⭐ Advanced

### Approach
Rather than using an LLM to evaluate another LLM's output (which is expensive
and non-deterministic), this exercise implements a **rule-based quality rubric**
with four binary checks, each worth 1 point (maximum score 4).

The checks enforce the schema contract that the Gemini prompt in the capstone
is supposed to produce:

| Check | What it validates |
|-------|-------------------|
| `has_steps` | Exactly 4 remediation steps — matches the prompt's instruction |
| `has_commands` | Every step has a Nutanix CLI tool — ensures actionability |
| `has_severity` | Severity is P1/P2/P3 — matches Nutanix incident taxonomy |
| `has_root_cause` | Non-empty root cause string — ensures explainability |

Checking for *exactly* one of `{'ncli', 'acli', 'ncc', 'allssh', 'genesis'}` as
a substring of the `command` field is intentionally permissive — a command like
`'genesis stop; genesis start'` still passes because it contains `genesis`.

In [33]:
NUTANIX_CLI_TOOLS = {'ncli', 'acli', 'ncc', 'allssh', 'genesis'}


def score_remediation(remediation: dict) -> dict:
    """
    Evaluate the quality of a Gemini-generated remediation plan.

    Parameters
    ----------
    remediation : dict
        Expected keys: severity, root_cause, remediation_steps (list of dicts
        each having 'command').

    Returns
    -------
    dict with boolean checks and an integer score 0–4.
    """
    steps = remediation.get('remediation_steps', [])

    # Check 1 — exactly 4 steps
    has_steps = isinstance(steps, list) and len(steps) == 4

    # Check 2 — every step contains a Nutanix CLI tool in its 'command' field
    if has_steps:
        has_commands = all(
            any(tool in step.get('command', '') for tool in NUTANIX_CLI_TOOLS)
            for step in steps
        )
    else:
        # Can't validate commands if steps are malformed
        has_commands = False

    # Check 3 — severity is a valid Nutanix incident priority
    has_severity = remediation.get('severity', '') in {'P1', 'P2', 'P3'}

    # Check 4 — root_cause is a non-empty, non-whitespace string
    has_root_cause = bool(str(remediation.get('root_cause', '')).strip())

    checks = dict(
        has_steps     = has_steps,
        has_commands  = has_commands,
        has_severity  = has_severity,
        has_root_cause= has_root_cause,
    )
    checks['score'] = int(sum(checks.values()))   # 0–4
    return checks


# ── Test fixtures ─────────────────────────────────────────────────────────
good_remediation = {
    'severity': 'P1',
    'root_cause': 'CPU overcommit caused by simultaneous VM migration on node-1.',
    'remediation_steps': [
        {'step': 1, 'action': 'Check CPU ready times',
         'command': 'acli host.get node-1'},
        {'step': 2, 'action': 'Live-migrate VMs to less-loaded node',
         'command': 'acli vm.migrate vm_name=webserver01 host=node-2'},
        {'step': 3, 'action': 'Restart Stargate to clear backpressure',
         'command': 'genesis stop stargate; genesis start'},
        {'step': 4, 'action': 'Validate cluster health post-migration',
         'command': 'ncli cluster health-check'},
    ],
}

bad_remediation = {
    'severity': 'HIGH',   # invalid — not P1/P2/P3
    'root_cause': '',     # empty string
    'remediation_steps': [],  # no steps
}

partial_remediation = {
    'severity': 'P2',
    'root_cause': 'Memory pressure from Cassandra.',
    'remediation_steps': [
        {'step': 1, 'action': 'Restart Cassandra', 'command': 'restart_cassandra.sh'},  # no CLI tool
        {'step': 2, 'action': 'Check heap',         'command': 'jstack 1234'},
        {'step': 3, 'action': 'Check cluster',      'command': 'ncc health_checks run_all'},
        {'step': 4, 'action': 'Alert oncall',        'command': 'echo alert'},
    ],
}

# ── Acceptance tests ──────────────────────────────────────────────────────
good_score    = score_remediation(good_remediation)
bad_score     = score_remediation(bad_remediation)
partial_score = score_remediation(partial_remediation)

assert good_score['score']    == 4, f"Good should score 4, got {good_score['score']}"
assert bad_score['score']     == 0, f"Bad should score 0, got {bad_score['score']}"
assert 1 <= partial_score['score'] <= 3, \
    f"Partial should score 1–3, got {partial_score['score']}"

print('Exercise 5 passed ✅')
print(f'Good remediation   : {good_score}')
print(f'Bad remediation    : {bad_score}')
print(f'Partial remediation: {partial_score}')

Exercise 5 passed ✅
Good remediation   : {'has_steps': True, 'has_commands': True, 'has_severity': True, 'has_root_cause': True, 'score': 4}
Bad remediation    : {'has_steps': False, 'has_commands': False, 'has_severity': False, 'has_root_cause': False, 'score': 0}
Partial remediation: {'has_steps': True, 'has_commands': False, 'has_severity': True, 'has_root_cause': True, 'score': 3}


---
## Exercise 6 — Audit Log Utilities
**Difficulty:** ⭐⭐ Intermediate

### Approach
**`load_audit`** reads the JSONL file line-by-line (one JSON object per line)
and uses `pd.json_normalize` to flatten nested dicts such as
`{"detection": {"is_anomaly": true}}` into a flat column
`detection.is_anomaly`. This handles both the flat format used in simple
pipelines and the nested format the capstone writes.

**`audit_summary`** uses a column-name resolution strategy: it tries the
dotted nested form first (`detection.is_anomaly`), then the flat form
(`is_anomaly`). This makes both functions robust to either schema. Key metrics:
- `anomaly_rate` — percentage of events that were anomalies
- `severity_counts` — `value_counts()` on the severity column
- `most_affected_node` — the node appearing most in anomaly rows (useful for
  identifying a rogue node that is consistently unhealthy)

In [34]:
def load_audit(path: str) -> pd.DataFrame:
    """
    Read a JSONL audit log and return a flat DataFrame.

    Handles both flat records and nested dicts via json_normalize.
    Returns an empty DataFrame if the file is absent or empty.
    """
    p = pathlib.Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame()

    rows = [
        json.loads(line)
        for line in p.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    if not rows:
        return pd.DataFrame()

    return pd.json_normalize(rows)


def audit_summary(df: pd.DataFrame) -> dict:
    """
    Summarise a loaded audit log DataFrame.

    Returns
    -------
    dict with keys:
        total_events       : int
        anomaly_rate       : float (percentage)
        severity_counts    : dict {severity: count}
        most_affected_node : str | None
    """
    if df.empty:
        return {
            'total_events':       0,
            'anomaly_rate':       0.0,
            'severity_counts':    {},
            'most_affected_node': None,
        }

    # Column name resolution: support nested (json_normalize) and flat schemas
    def resolve(candidates):
        return next((c for c in candidates if c in df.columns), None)

    anom_col = resolve(['detection.is_anomaly', 'is_anomaly'])
    sev_col  = resolve(['detection.severity',   'severity'])
    node_col = resolve(['detection.node',        'node'])

    anomalies  = int(df[anom_col].sum()) if anom_col else 0
    sev_counts = df[sev_col].value_counts().to_dict() if sev_col else {}

    if node_col and anom_col and anomalies > 0:
        anom_nodes = df.loc[df[anom_col] == True, node_col]
        most_node  = anom_nodes.value_counts().index[0] if len(anom_nodes) else None
    else:
        most_node = None

    return {
        'total_events':       len(df),
        'anomaly_rate':       round(anomalies / len(df) * 100, 1),
        'severity_counts':    sev_counts,
        'most_affected_node': most_node,
    }


# ── Test with synthetic data ──────────────────────────────────────────────
import tempfile, os

synthetic_records = [
    {'timestamp': '2025-06-01T10:00:00Z', 'node': 'node-1',
     'is_anomaly': True,  'severity': 'critical'},
    {'timestamp': '2025-06-01T10:01:00Z', 'node': 'node-2',
     'is_anomaly': False, 'severity': 'none'},
    {'timestamp': '2025-06-01T10:02:00Z', 'node': 'node-1',
     'is_anomaly': True,  'severity': 'warning'},
    {'timestamp': '2025-06-01T10:03:00Z', 'node': 'node-3',
     'is_anomaly': False, 'severity': 'none'},
    {'timestamp': '2025-06-01T10:04:00Z', 'node': 'node-1',
     'is_anomaly': True,  'severity': 'critical'},
]

with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl',
                                  delete=False) as f:
    for rec in synthetic_records:
        f.write(json.dumps(rec) + '\n')
    tmp_path = f.name

df_audit = load_audit(tmp_path)
summary  = audit_summary(df_audit)
os.unlink(tmp_path)

assert summary['total_events']       == 5,       'total_events should be 5'
assert summary['anomaly_rate']       == 60.0,    'anomaly_rate should be 60%'
assert summary['most_affected_node'] == 'node-1','node-1 has most anomalies'

print('Exercise 6 passed ✅')
print(f"  Total events    : {summary['total_events']}")
print(f"  Anomaly rate    : {summary['anomaly_rate']}%")
print(f"  Severity counts : {summary['severity_counts']}")
print(f"  Most affected   : {summary['most_affected_node']}")

# ── Also try the real audit log if it exists ──────────────────────────────
real_path = pathlib.Path('/Users/nikhil/AI:ML intermediate/Module_7/aiops_audit.jsonl')
if real_path.exists() and real_path.stat().st_size > 0:
    df_real   = load_audit(str(real_path))
    real_sum  = audit_summary(df_real)
    print(f'\nReal audit log ({len(df_real)} events):')
    for k, v in real_sum.items():
        print(f'  {k:22}: {v}')
else:
    print('\n(Real audit log not found — run the capstone pipeline to generate it.)')

Exercise 6 passed ✅
  Total events    : 5
  Anomaly rate    : 60.0%
  Severity counts : {'critical': 2, 'none': 2, 'warning': 1}
  Most affected   : node-1

Real audit log (5 events):
  total_events          : 5
  anomaly_rate          : 100.0
  severity_counts       : {'critical': 5}
  most_affected_node    : node-1


---
## Exercise 7 — End-to-End Mini Pipeline
**Difficulty:** ⭐⭐⭐ Advanced

### Approach
`mini_pipeline` replicates the capstone's 4-step pipeline in a single callable
function, making it easy to unit-test, call from a REST endpoint, or run in
parallel across nodes (Exercise 8).

**Step 1 — Feature engineering:** computes `disk_pressure` and `network_cpu_ratio`
inline from the raw metrics dict. No DataFrame copy is needed because we are
working with a single row.

**Step 2 — Anomaly detection:** applies the same `scaler` fitted in Exercise 3,
then calls both `predict` (binary label) and `score_samples` (continuous score).

**Step 3 — Severity classification:** uses `model_strict.offset_` as a reference
point. Scores further below the offset → higher confidence → higher severity.
The confidence formula clips to [0, 1] so it is always a valid probability-like
value for downstream consumers.

**Step 4 — Audit entry:** builds a serialisable dict without writing to disk,
giving callers the flexibility to batch writes or send to a message queue.

In [35]:
def mini_pipeline(metrics: dict, node: str) -> dict:
    """
    Run the full 4-step AIOps pipeline for a single node.

    Steps
    -----
    1. Feature engineering   — add disk_pressure and network_cpu_ratio
    2. Anomaly detection     — IsolationForest score + binary label
    3. Severity classification — critical / warning / info / none
    4. Audit entry           — build (do NOT write) a serialisable audit dict

    Parameters
    ----------
    metrics : dict  Raw CVM metric values (must contain all FEATURES except
                    disk_pressure and network_cpu_ratio, which are derived).
    node    : str   Node identifier, e.g. 'node-1'.

    Returns
    -------
    dict with keys:
        node, is_anomaly, severity, confidence,
        feature_contributions, audit_entry
    """
    # ── Step 1: feature engineering ───────────────────────────────────────
    m = metrics.copy()
    m['disk_pressure']     = (
        (m['disk_read_mbps'] + m['disk_write_mbps'])
        / (m.get('stargate_ops_per_sec', 1) + 0.001)
    )
    m['network_cpu_ratio'] = m['network_mbps'] / (m['cpu_pct'] + 1)

    # ── Step 2: anomaly detection ─────────────────────────────────────────
    X     = np.array([[m[f] for f in FEATURES]])
    X_s   = scaler.transform(X)
    score = float(model_strict.score_samples(X_s)[0])
    is_an = bool(model_strict.predict(X_s)[0] == -1)

    # ── Step 3: severity classification ──────────────────────────────────
    # Normalise the raw score relative to the model's decision boundary
    offset = model_strict.offset_
    conf   = float(np.clip(
        1.0 - (score - offset) / (abs(offset) + 1e-9),
        0.0, 1.0
    ))
    severity = (
        'critical' if is_an and conf > 0.85 else
        'warning'  if is_an and conf > 0.60 else
        'info'     if is_an else
        'none'
    )

    # ── Top-3 feature contributions (by absolute z-score) ────────────────
    z      = X_s[0]
    top_ix = sorted(range(len(FEATURES)),
                    key=lambda i: abs(z[i]), reverse=True)[:3]
    contributions = {FEATURES[i]: round(float(z[i]), 4) for i in top_ix}

    # ── Step 4: audit entry (not written to disk here) ────────────────────
    audit_entry = {
        'timestamp':             datetime.utcnow().isoformat() + 'Z',
        'node':                  node,
        'is_anomaly':            is_an,
        'severity':              severity,
        'confidence':            round(conf, 4),
        'anomaly_score':         round(score, 5),
        'feature_contributions': contributions,
        'metrics':               metrics,
    }

    return dict(
        node                 = node,
        is_anomaly           = is_an,
        severity             = severity,
        confidence           = round(conf, 4),
        feature_contributions= contributions,
        audit_entry          = audit_entry,
    )


# ── Acceptance tests ──────────────────────────────────────────────────────
normal_result = mini_pipeline(
    dict(cpu_pct=35,  mem_pct=55,  iops=8_000,  latency_ms=2.5,
         disk_read_mbps=120, disk_write_mbps=80,
         network_mbps=500,   stargate_ops_per_sec=2_000),
    'node-1'
)

anomaly_result = mini_pipeline(
    dict(cpu_pct=94,  mem_pct=91,  iops=19_500, latency_ms=42,
         disk_read_mbps=118, disk_write_mbps=79,
         network_mbps=495,   stargate_ops_per_sec=350),
    'node-3'
)

assert not normal_result['is_anomaly'],  'Normal metrics should not be an anomaly'
assert anomaly_result['is_anomaly'],     'Spike metrics should be flagged as anomaly'
assert anomaly_result['severity'] in {'critical', 'warning'}, \
    f"Spike severity should be critical/warning, got {anomaly_result['severity']}"
assert 'audit_entry' in normal_result,  'Result must include audit_entry'
assert len(normal_result['feature_contributions'])  == 3, 'Exactly 3 contributions'
assert len(anomaly_result['feature_contributions']) == 3, 'Exactly 3 contributions'

print('Exercise 7 passed ✅')
print(f"  Normal  — anomaly={normal_result['is_anomaly']}  "
      f"severity={normal_result['severity']}  "
      f"confidence={normal_result['confidence']}")
print(f"  Anomaly — anomaly={anomaly_result['is_anomaly']}  "
      f"severity={anomaly_result['severity']}  "
      f"confidence={anomaly_result['confidence']}")
print(f"  Top features (anomaly): {anomaly_result['feature_contributions']}")

Exercise 7 passed ✅
  Normal  — anomaly=False  severity=none  confidence=0.7983
  Anomaly — anomaly=True  severity=critical  confidence=1.0
  Top features (anomaly): {'latency_ms': 33.7638, 'disk_pressure': 14.0956, 'cpu_pct': 6.7095}


/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


---
## Exercise 8 — Multi-Node Parallel Batch Run
**Difficulty:** ⭐⭐⭐ Advanced

### Approach
`ThreadPoolExecutor` is the right tool here (not `ProcessPoolExecutor`) because
the bottleneck in a real deployment would be I/O — fetching live metrics from
Prism or Prometheus — not CPU. Threads allow concurrent I/O without the
serialisation overhead of multiprocessing.

`as_completed` is used to collect results as they finish rather than waiting
for all of them in submission order. For equal-cost tasks this makes little
practical difference, but it is the idiomatic pattern because it handles
slow or failing nodes gracefully.

The five test scenarios are deliberately diverse:
- `node-1`: healthy baseline
- `node-2`: CPU/memory/ops triple-spike (clear anomaly)
- `node-3`: disk write saturation + very high IOPS + latency
- `node-4`: network storm signature
- `node-5`: memory exhaustion + very low Stargate ops

In [36]:
TEST_NODES = [
    ('node-1', dict(cpu_pct=35,  mem_pct=55,  iops=8_000, latency_ms=2.5,
                    disk_read_mbps=120, disk_write_mbps=80,
                    network_mbps=500,   stargate_ops_per_sec=2_000)),
    ('node-2', dict(cpu_pct=94,  mem_pct=91,  iops=19_500, latency_ms=42,
                    disk_read_mbps=118, disk_write_mbps=79,
                    network_mbps=495,   stargate_ops_per_sec=350)),
    ('node-3', dict(cpu_pct=58,  mem_pct=64,  iops=24_000, latency_ms=115,
                    disk_read_mbps=115, disk_write_mbps=482,
                    network_mbps=510,   stargate_ops_per_sec=1_920)),
    ('node-4', dict(cpu_pct=71,  mem_pct=59,  iops=7_900,  latency_ms=29,
                    disk_read_mbps=112, disk_write_mbps=76,
                    network_mbps=1_960, stargate_ops_per_sec=1_800)),
    ('node-5', dict(cpu_pct=44,  mem_pct=98,  iops=3_150,  latency_ms=80,
                    disk_read_mbps=106, disk_write_mbps=81,
                    network_mbps=515,   stargate_ops_per_sec=87)),
]


def run_node(args):
    """Worker function: unpack (node, metrics) tuple and call mini_pipeline."""
    node, metrics = args
    return mini_pipeline(metrics, node)


# ── Run all 5 nodes in parallel with 3 workers ────────────────────────────
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(run_node, (node, metrics)): node
        for node, metrics in TEST_NODES
    }
    batch_results = [future.result() for future in as_completed(futures)]

# Sort by node name for a consistent display
batch_results.sort(key=lambda r: r['node'])

# ── Print summary table ───────────────────────────────────────────────────
print(f"{'Node':8}  {'Anomaly':8}  {'Severity':10}  {'Confidence':11}  Top Feature")
print('-' * 68)
for r in batch_results:
    top_feat = max(
        r['feature_contributions'],
        key=lambda k: abs(r['feature_contributions'][k])
    )
    print(f"  {r['node']:6}  {str(r['is_anomaly']):8}  {r['severity']:10}  "
          f"{r['confidence']:.4f}       {top_feat}")

anomaly_count = sum(1 for r in batch_results if r['is_anomaly'])
print(f"\nTotal anomalies detected: {anomaly_count}/{len(batch_results)}")

assert anomaly_count >= 2, (
    f'Expected at least 2 anomalies in test scenarios, got {anomaly_count}'
)
print('Exercise 8 passed ✅')

Node      Anomaly   Severity    Confidence   Top Feature
--------------------------------------------------------------------
  node-1  False     none        0.7983       disk_pressure
  node-2  True      critical    1.0000       latency_ms
  node-3  True      critical    1.0000       latency_ms
  node-4  True      critical    1.0000       latency_ms
  node-5  True      critical    1.0000       latency_ms

Total anomalies detected: 4/5
Exercise 8 passed ✅


/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have va

---
## Exercise 9 — RAG Runbook Retrieval
**Difficulty:** ⭐⭐⭐⭐ Expert

### Approach

#### Part A — Keyword matching (required)
Each KB article already has a `keywords` set. The retriever extracts search
terms from the detection result by splitting feature names on underscores
(e.g., `disk_write_mbps` → `{'disk', 'write', 'mbps'}`) and adds the severity
label. The score for each article is the size of the intersection between the
search terms and the article's keywords — a fast, deterministic, zero-dependency
approach that works well when keywords are carefully curated.

#### Part B — TF-IDF cosine similarity (stretch)
For richer matching, `TfidfVectorizer` converts the article content + title +
keywords into TF-IDF vectors. The query is built from the detection's top
features and severity. `cosine_similarity` finds articles whose vocabulary
overlap with the query terms, going beyond exact keyword matches to handle
synonyms and related terms in the article text.

In [37]:
# ── KB Article corpus ─────────────────────────────────────────────────────
KB_ARTICLES = [
    {
        'id': 'NX-KB-2201',
        'title': 'Stargate WAL Corruption Recovery',
        'keywords': {'stargate', 'wal', 'crash', 'disk', 'stargate_health', 'io'},
        'content': 'Symptoms: stargate_ops_per_sec < 200, WAL write failure.',
    },
    {
        'id': 'NX-KB-1845',
        'title': 'Disk I/O Saturation — Erasure Coding Rebuild',
        'keywords': {'disk', 'io', 'iops', 'latency', 'rebuild',
                     'disk_write_mbps', 'io_saturation'},
        'content': 'Symptoms: disk_write_mbps > 400, iops > 20000, latency_ms > 50ms.',
    },
    {
        'id': 'NX-KB-3102',
        'title': 'Network Storm Isolation on OVS Bridge',
        'keywords': {'network', 'storm', 'ovs', 'network_mbps', 'broadcast', 'lacp'},
        'content': 'Symptoms: network_mbps > 1500.',
    },
    {
        'id': 'NX-KB-2756',
        'title': 'CVM Memory Exhaustion — Cassandra OOM Recovery',
        'keywords': {'memory', 'mem_pct', 'cassandra', 'oom', 'cvm',
                     'stargate_health'},
        'content': 'Symptoms: mem_pct > 95%, stargate_ops_per_sec < 200.',
    },
    {
        'id': 'NX-KB-4001',
        'title': 'CPU Overcommit — AHV VM Migration',
        'keywords': {'cpu', 'overcommit', 'cpu_pct', 'mem_pct',
                     'cpu_mem_pressure', 'vm'},
        'content': 'Symptoms: cpu_pct > 90% sustained.',
    },
]


# ── Part A: keyword matching ──────────────────────────────────────────────
def find_kb_articles(
    detection: dict,
    kb: list,
    top_k: int = 2
) -> list:
    """
    Retrieve the most relevant KB articles for a detection result using
    keyword intersection scoring.

    The search vocabulary is built from:
    - Individual tokens from top feature names (split on '_')
    - The full feature names themselves (handles compound keys like 'disk_write_mbps')
    - The severity label

    Parameters
    ----------
    detection : dict   Output of mini_pipeline or similar, with
                       'feature_contributions' and 'severity' keys.
    kb        : list   List of KB article dicts with 'keywords' sets.
    top_k     : int    Number of articles to return.

    Returns
    -------
    List of up to top_k article dicts, sorted by score descending.
    Articles with score 0 are excluded.
    """
    search_terms: set = set()

    for feat in detection.get('feature_contributions', {}):
        # Add full feature name (e.g., 'disk_write_mbps') and individual tokens
        search_terms.add(feat.lower())
        search_terms.update(feat.lower().split('_'))

    search_terms.add(detection.get('severity', '').lower())

    scored = [
        (len(search_terms & article['keywords']), article)
        for article in kb
    ]
    scored.sort(key=lambda x: x[0], reverse=True)

    return [article for score, article in scored[:top_k] if score > 0]


# ── Part B: TF-IDF cosine similarity (stretch) ────────────────────────────
def find_kb_articles_tfidf(
    detection: dict,
    kb: list,
    top_k: int = 2
) -> list:
    """
    Retrieve KB articles using TF-IDF cosine similarity.

    Each article is represented as: title + keywords + content.
    The query is built from the detection's top feature names + severity.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    # Build document strings for each article
    docs = [
        f"{a['title']} {' '.join(a['keywords'])} {a['content']}"
        for a in kb
    ]

    # Build query string from detection
    features   = ' '.join(detection.get('feature_contributions', {}).keys())
    features_t = features.replace('_', ' ')  # tokenise compound names
    severity   = detection.get('severity', '')
    query      = f"{features} {features_t} {severity}"

    vectorizer = TfidfVectorizer()
    tfidf_mat  = vectorizer.fit_transform(docs)
    query_vec  = vectorizer.transform([query])

    scores     = cosine_similarity(query_vec, tfidf_mat).flatten()
    top_idx    = scores.argsort()[::-1][:top_k]

    return [kb[i] for i in top_idx if scores[i] > 0]


# ── Acceptance tests ──────────────────────────────────────────────────────
disk_detection = {
    'feature_contributions': {
        'disk_write_mbps': 4.2,
        'io_saturation':   3.1,
        'iops':            2.8,
    },
    'severity': 'critical',
    'node': 'node-3',
}
cpu_detection = {
    'feature_contributions': {
        'cpu_pct':          5.1,
        'cpu_mem_pressure': 4.3,
        'mem_pct':          3.2,
    },
    'severity': 'critical',
    'node': 'node-1',
}
net_detection = {
    'feature_contributions': {
        'network_mbps':       6.0,
        'network_cpu_ratio':  5.5,
        'latency_ms':         2.1,
    },
    'severity': 'warning',
    'node': 'node-4',
}

disk_articles = find_kb_articles(disk_detection, KB_ARTICLES)
cpu_articles  = find_kb_articles(cpu_detection,  KB_ARTICLES)
net_articles  = find_kb_articles(net_detection,  KB_ARTICLES)

assert len(disk_articles) > 0, 'Should find at least 1 article for disk anomaly'
assert len(cpu_articles)  > 0, 'Should find at least 1 article for CPU anomaly'
assert len(net_articles)  > 0, 'Should find at least 1 article for network anomaly'

# Verify relevance: disk anomaly should not return the CPU article as #1
assert disk_articles[0]['id'] != 'NX-KB-4001', \
    'Top result for disk anomaly should not be the CPU article'

print('Exercise 9 passed ✅')
print('\n── Keyword matching ─────────────────────────────────────────────────')
for label, arts in [('Disk anomaly', disk_articles),
                     ('CPU anomaly',  cpu_articles),
                     ('Net anomaly',  net_articles)]:
    print(f"  {label}:")
    for a in arts:
        print(f"    {a['id']} — {a['title']}")

print('\n── TF-IDF cosine similarity (stretch) ───────────────────────────────')
for label, det in [('Disk anomaly', disk_detection),
                    ('CPU anomaly',  cpu_detection),
                    ('Net anomaly',  net_detection)]:
    tfidf_arts = find_kb_articles_tfidf(det, KB_ARTICLES)
    print(f"  {label}:")
    for a in tfidf_arts:
        print(f"    {a['id']} — {a['title']}")

Exercise 9 passed ✅

── Keyword matching ─────────────────────────────────────────────────
  Disk anomaly:
    NX-KB-1845 — Disk I/O Saturation — Erasure Coding Rebuild
    NX-KB-2201 — Stargate WAL Corruption Recovery
  CPU anomaly:
    NX-KB-4001 — CPU Overcommit — AHV VM Migration
    NX-KB-2756 — CVM Memory Exhaustion — Cassandra OOM Recovery
  Net anomaly:
    NX-KB-3102 — Network Storm Isolation on OVS Bridge
    NX-KB-1845 — Disk I/O Saturation — Erasure Coding Rebuild

── TF-IDF cosine similarity (stretch) ───────────────────────────────
  Disk anomaly:
    NX-KB-1845 — Disk I/O Saturation — Erasure Coding Rebuild
    NX-KB-2201 — Stargate WAL Corruption Recovery
  CPU anomaly:
    NX-KB-4001 — CPU Overcommit — AHV VM Migration
    NX-KB-2756 — CVM Memory Exhaustion — Cassandra OOM Recovery
  Net anomaly:
    NX-KB-3102 — Network Storm Isolation on OVS Bridge
    NX-KB-4001 — CPU Overcommit — AHV VM Migration


---
## Exercise 10 — Full Pipeline Integration Test
**Difficulty:** ⭐⭐⭐⭐ Expert

### Approach
A good integration test validates behaviour across an end-to-end slice of the
system, not just individual units. This test generates a realistic mixed
dataset (20 normal + 5 anomalous rows), runs `mini_pipeline` on every row, and
validates four orthogonal properties:

| Check | What it validates |
|-------|-------------------|
| `anomaly_rate_in_range` | Model sensitivity is reasonable (not 0% or 100%) |
| `valid_severities` | Classifier only produces known severity labels |
| `three_contributions` | Feature importance extraction always returns exactly 3 |
| `valid_audit_timestamps` | Audit entries have parseable ISO 8601 timestamps |

The anomaly rate bounds (5–60%) are deliberately wide to account for the
fact that `model_strict` uses 2% contamination — it may only flag 1-2 of the
5 spike rows as anomalies, giving a 4–8% rate. The upper bound of 60% prevents
a misconfigured model from passing by flagging everything.

Each check prints `PASS` or `FAIL` so the test report is immediately readable
in a notebook or CI log.

In [38]:
def test_pipeline_integration() -> bool:
    """
    Integration test for the complete AIOps pipeline.

    Generates 20 normal + 5 anomalous CVM rows, runs mini_pipeline on each,
    and validates four checks. Returns True if all checks pass.
    """
    checks: Dict[str, bool] = {}

    # ── Generate test data ────────────────────────────────────────────────
    df_n = generate_scenario('normal',    20)
    df_a = generate_scenario('cpu_spike',  5)
    all_rows = pd.concat([df_n, df_a], ignore_index=True)

    # ── Run pipeline on every row ─────────────────────────────────────────
    results = []
    for _, row in all_rows.iterrows():
        m = row.drop('node').to_dict()
        r = mini_pipeline(m, row['node'])
        results.append(r)

    total     = len(results)
    n_anomaly = sum(r['is_anomaly'] for r in results)
    anom_rate = n_anomaly / total

    # ── Check 1: anomaly rate is in a reasonable range ────────────────────
    # Wide bounds: model_strict(2%) may flag 1-2 of 5 spikes; upper bound
    # prevents a trivially over-flagging model from passing.
    checks['anomaly_rate_in_range'] = 0.05 <= anom_rate <= 0.60

    # ── Check 2: all severity values are valid ────────────────────────────
    valid_severities = {'critical', 'warning', 'info', 'none'}
    checks['valid_severities'] = all(
        r['severity'] in valid_severities for r in results
    )

    # ── Check 3: exactly 3 feature contributions per result ───────────────
    checks['three_contributions'] = all(
        len(r['feature_contributions']) == 3 for r in results
    )

    # ── Check 4: every audit entry has a parseable ISO 8601 timestamp ─────
    def is_valid_iso(ts: str) -> bool:
        try:
            datetime.fromisoformat(ts.replace('Z', '+00:00'))
            return True
        except (ValueError, AttributeError):
            return False

    checks['valid_audit_timestamps'] = all(
        is_valid_iso(r['audit_entry']['timestamp']) for r in results
    )

    # ── Print report ──────────────────────────────────────────────────────
    print(f"{'Check':<30}  {'Result'}")
    print('-' * 42)
    all_pass = True
    for check, passed in checks.items():
        status   = 'PASS ✅' if passed else 'FAIL ❌'
        all_pass = all_pass and passed
        print(f"  {check:<30}  {status}")

    print()
    print(f"  Rows processed   : {total}")
    print(f"  Anomalies flagged: {n_anomaly} ({anom_rate:.1%})")
    print(f"  Severity breakdown: "
          + ', '.join(
              f"{sev}={sum(1 for r in results if r['severity'] == sev)}"
              for sev in ('critical', 'warning', 'info', 'none')
          ))

    # ── Detailed diagnostics for failing checks ───────────────────────────
    if not checks['anomaly_rate_in_range']:
        print(f"\n  [FAIL] anomaly_rate={anom_rate:.1%} "
              f"(expected 5%–60%)")
    if not checks['valid_severities']:
        bad = {r['severity'] for r in results
               if r['severity'] not in valid_severities}
        print(f"\n  [FAIL] Unknown severity values: {bad}")
    if not checks['three_contributions']:
        bad_counts = {len(r['feature_contributions']) for r in results
                      if len(r['feature_contributions']) != 3}
        print(f"\n  [FAIL] Unexpected contribution counts: {bad_counts}")
    if not checks['valid_audit_timestamps']:
        bad_ts = [
            r['audit_entry']['timestamp'] for r in results
            if not is_valid_iso(r['audit_entry']['timestamp'])
        ][:3]
        print(f"\n  [FAIL] Invalid timestamps (first 3): {bad_ts}")

    print()
    if all_pass:
        print('All checks PASSED ✅ — Exercise 10 passed ✅')
    else:
        print('Some checks FAILED ❌')

    return all_pass


# ── Run the integration test ──────────────────────────────────────────────
test_pipeline_integration()

Check                           Result
------------------------------------------
  anomaly_rate_in_range           PASS ✅
  valid_severities                PASS ✅
  three_contributions             PASS ✅
  valid_audit_timestamps          PASS ✅

  Rows processed   : 25
  Anomalies flagged: 8 (32.0%)
  Severity breakdown: critical=8, warning=0, info=0, none=17

All checks PASSED ✅ — Exercise 10 passed ✅


/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/nikhil/AI:ML intermediate/myenv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have va

True

---
## Summary

| Exercise | Topic | Difficulty | Key Concepts |
|----------|-------|-----------|-------------|
| 1 | Telemetry generation | ⭐ | `np.random.uniform`, parameterised scenarios |
| 2 | Feature engineering | ⭐⭐ | Ratio features, zero-division guards |
| 3 | Anomaly detection tuning | ⭐⭐ | `IsolationForest` contamination, `offset_` |
| 4 | FastAPI endpoint design | ⭐⭐ | Pydantic, batch REST patterns |
| 5 | LLM output quality scoring | ⭐⭐⭐ | Rule-based rubrics, schema validation |
| 6 | Audit log utilities | ⭐⭐ | JSONL I/O, `json_normalize`, flexible column resolution |
| 7 | Full pipeline function | ⭐⭐⭐ | End-to-end composition, confidence scoring |
| 8 | Parallel multi-node run | ⭐⭐⭐ | `ThreadPoolExecutor`, `as_completed` |
| 9 | RAG runbook retrieval | ⭐⭐⭐⭐ | Keyword intersection, TF-IDF cosine similarity |
| 10 | Integration test | ⭐⭐⭐⭐ | End-to-end validation, diagnostic reporting |

### Gemini API Note
All exercises in this solutions notebook use **`gemini-2.0-flash-lite`** (not `gemini-1.5-flash`).
In the capstone API calls, use:
```python
model = genai.GenerativeModel('gemini-2.0-flash-lite')
```

Well done — you have built, tested, and validated every layer of a
production AIOps pipeline: raw telemetry ingestion, feature engineering,
ML-based anomaly detection, severity classification, LLM-generated remediation,
audit logging, parallel execution, and retrieval-augmented runbook lookup.

---
## Troubleshooting Guide

Common issues you may encounter and how to resolve them.


In [39]:
# ── Troubleshooting: run this cell if any exercise fails ─────────────────
import os, sys, importlib

ISSUES = {}

# 1. Check Gemini API key
gemini_key = os.environ.get('GEMINI_API_KEY', '')
ISSUES['gemini_api_key'] = (
    'OK — key loaded' if gemini_key
    else 'MISSING — add GEMINI_API_KEY to your .env file or run: '
         'os.environ["GEMINI_API_KEY"] = "AIza..."'
)

# 2. Check that models are trained on normal-only data
try:
    offset = model_strict.offset_
    if offset < -0.6:
        ISSUES['model_offset'] = (
            f'WARNING — model_strict.offset_={offset:.4f} is very low. '
            'Models may have been trained on mixed data. '
            'Re-run the "Retrain on NORMAL-ONLY data" cell above.'
        )
    else:
        ISSUES['model_offset'] = f'OK — model_strict.offset_={offset:.4f}'
except NameError:
    ISSUES['model_offset'] = 'ERROR — model_strict not defined. Run cell 8 then the normal-only cell.'

# 3. Check key imports
for pkg in ['numpy', 'pandas', 'sklearn', 'pydantic', 'google.generativeai']:
    try:
        importlib.import_module(pkg)
        ISSUES[f'import_{pkg}'] = 'OK'
    except ImportError:
        ISSUES[f'import_{pkg}'] = f'MISSING — run: pip install {pkg.replace(".", "-")}'

# 4. Check audit log
audit_path = pathlib.Path('./aiops_audit.jsonl')
parent_audit = pathlib.Path('../aiops_audit.jsonl')
if audit_path.exists():
    ISSUES['audit_log'] = f'OK — {audit_path} ({audit_path.stat().st_size} bytes)'
elif parent_audit.exists():
    ISSUES['audit_log'] = f'OK — found at {parent_audit}'
else:
    ISSUES['audit_log'] = 'INFO — no audit log yet; it is created when mini_pipeline writes entries.'

# 5. Verify FEATURES list
try:
    missing_features = [f for f in FEATURES if f not in df_normal_prod.columns]
    ISSUES['features'] = (
        'OK' if not missing_features
        else f'MISSING columns: {missing_features} — re-run add_custom_features'
    )
except NameError:
    ISSUES['features'] = 'INFO — df_normal_prod not yet defined; run the normal-only training cell.'

print('\n=== Troubleshooting Report ===')
for k, v in ISSUES.items():
    status = 'OK' if v.startswith('OK') else ('WARN' if v.startswith('WARN') or v.startswith('INFO') else 'FAIL')
    icon = {'OK': '✅', 'WARN': '⚠️', 'FAIL': '❌'}.get(status, '?')
    print(f'  {icon}  {k:<25} {v}')
print()
print('Fix any FAIL items above, then re-run the failing exercise cell.')



=== Troubleshooting Report ===
  ✅  gemini_api_key            OK — key loaded
  ✅  model_offset              OK — model_strict.offset_=-0.5450
  ✅  import_numpy              OK
  ✅  import_pandas             OK
  ✅  import_sklearn            OK
  ✅  import_pydantic           OK
  ✅  import_google.generativeai OK
  ✅  audit_log                 OK — aiops_audit.jsonl (3597 bytes)
  ✅  features                  OK

Fix any FAIL items above, then re-run the failing exercise cell.
